# Can a Computer Hear a Failing Machine?

**Speech & Audio Analytics · Session 3 of 4** — Trained on REAL factory recordings — healthy fans vs faulty ones — then asked to judge a new one

## How to use this notebook

You do not need to write any code. **Run each cell in order** (click it, then press **Shift + Enter**) and read what comes out. A couple of cells play sound — unmute your speakers.

- Run cells **in order**, top to bottom.
- If anything looks broken: **Runtime → Restart and run all**.

As you scroll you'll meet four little callouts: 🟦 a new word, 🟩 what a cell just did, 🟧 something to try, 🟪 the recap.

> 💡 **THE BIG IDEA**
>
> An experienced technician can walk past a machine and say *"that one's on its way out"* — just from the **sound**. A healthy machine hums steadily; a failing one develops a whine, a rattle, a wobble.
> 
> Can a computer learn the same? **Yes** — and it is the *exact same idea* as every other predictive example: turn each thing into a few **numbers**, learn the pattern from past **labelled** examples, then judge a new one. Here the "thing" is a sound — and, crucially, we train on **real recordings**, not made-up ones, so nothing is rigged in the model's favour.

> 📍 **THE SCENARIO**
>
> **Our data is real.** We use the open **MIMII / DCASE** benchmark — actual microphone recordings of industrial **fans** running on a factory floor, each labelled **normal** or **faulty** (an unbalanced or clogged fan). Real recordings carry real noise, so there is **no tidy formula** that separates them — the model genuinely has to learn. We'll hear a healthy fan and a faulty one, turn the sound into numbers, train a model to tell them apart, then measure it honestly on recordings it never heard.

## Step 1 · Set up, and fetch the real recordings

> 🟦 **VOCABULARY — Sample rate**
>
> A sound is just air pressure measured many times a second. The **sample rate** is how many measurements per second — here **8000**. So a 3-second clip is really a list of about **24000 numbers**. To a computer, a sound *is already numbers*.

In [ ]:
import os, io, glob, zipfile, urllib.request, wave
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from IPython.display import Audio, display

# 120 real fan recordings (MIMII / DCASE, CC BY-SA) -- about 5 MB, fetched once.
# (raw.githubusercontent.com serves the file itself; a github.com/.../blob/... link
#  would return an HTML page, not a zip. Pinned to this branch until it is merged to main.)
URL = 'https://raw.githubusercontent.com/zerotoprod-5/HAL-AI-Tutorial/workshop-4session-realign/audio/fan_clips.zip'
if not os.path.isdir('fan_clips'):
    src = 'audio/fan_clips.zip' if os.path.exists('audio/fan_clips.zip') else 'fan_clips.zip'
    if not os.path.exists(src):
        print('Downloading real fan recordings (~5 MB) ...')
        urllib.request.urlretrieve(URL, src)
    zipfile.ZipFile(src).extractall('fan_clips')

files = sorted(glob.glob('fan_clips/*.wav'))
SR    = wave.open(files[0]).getframerate()
print('Got', len(files), 'real recordings at', SR, 'Hz.')

> 🟩 **WHAT JUST HAPPENED**
>
> We fetched **120 real fan recordings** — 60 healthy, 60 faulty — from the MIMII / DCASE benchmark. Each is a ~3-second clip. The normal/faulty tags are the **labels**, exactly like the answer column in every other notebook; only here the data is sound.

## Step 2 · Listen — a real healthy fan vs a real faulty one

Use your ears first. Press play on each — both are **real recordings**. Listen for the high **whine** the faulty fan develops on top of the hum:

In [ ]:
def load(path):
    w = wave.open(path)
    a = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(float)
    return a / 32768.0

clips  = [load(f) for f in files]
labels = np.array(['faulty' if 'anomaly' in os.path.basename(f) else 'normal' for f in files])

for state in ['normal', 'faulty']:
    i = list(labels).index(state)
    print('A REAL', state.upper(), 'fan:')
    display(Audio(clips[i], rate=SR))

> 🟩 **WHAT JUST HAPPENED**
>
> That whine in the faulty fan is real — no simulation. Your ear caught it in a second. But a plant has thousands of machines, and nobody can sit and listen to all of them. So next we turn each recording into numbers a computer can weigh.

## Step 3 · Turn each sound into a few numbers

> 🟦 **VOCABULARY — Features (from a sound)**
>
> We can't hand a model 24000 raw numbers and hope. We summarise each clip with a handful of **features** — the share of energy in each **frequency band** (low rumble … high whine), plus overall loudness and brightness. The **FFT** splits a sound into its pitches, like a prism splitting light, so we can measure them.

> ℹ️ **NOTE**
>
> The one line of maths is the **FFT**. You don't need the formula — think of it as a prism that tells you which pitches a sound is made of, so we can measure how much of each there is.

In [ ]:
def features(sig):
    spec  = np.abs(np.fft.rfft(sig * np.hanning(len(sig))))   # the pitches present
    freqs = np.fft.rfftfreq(len(sig), 1/SR)
    power = spec**2
    total = power.sum() + 1e-9
    edges = [0, 200, 500, 1000, 1800, 2800, 4000]            # frequency-band edges (Hz)
    feat  = {f'{lo}-{hi}Hz': float(power[(freqs >= lo) & (freqs < hi)].sum() / total)
             for lo, hi in zip(edges[:-1], edges[1:])}
    feat['loudness']   = float(np.sqrt(np.mean(sig**2)))     # overall loudness
    feat['brightness'] = float((freqs * power).sum() / total) # average pitch (Hz)
    return feat

feat = pd.DataFrame([features(c) for c in clips])
table = feat.copy(); table.insert(0, 'state', labels)
table.head()

> 🟩 **WHAT JUST HAPPENED**
>
> Every 3-second recording is now **one row of 8 numbers** — the share of energy in six frequency bands, plus loudness and brightness. A whiny faulty fan puts more energy in the high bands; that is the kind of pattern the model will key on.

## Step 4 · See the pattern with our own eyes

In [ ]:
plt.figure(figsize=(8, 5))
for state, color in [('normal', '#2e7d32'), ('faulty', '#b3433a')]:
    m = labels == state
    plt.scatter(feat.loc[m, '500-1000Hz'], feat.loc[m, '1800-2800Hz'],
                c=color, edgecolor='white', s=44, label=state, zorder=3)
plt.xlabel('energy in a LOW band (500-1000 Hz)')
plt.ylabel('energy in a HIGH band (1800-2800 Hz)')
plt.title('Real fans: healthy vs faulty'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

> 🟩 **WHAT JUST HAPPENED**
>
> The two colours lean apart — faulty fans ride higher on the high-band axis — but they **overlap**, because this is real data. There is no clean line to draw by hand. That overlap is exactly why we want a model that weighs *all* the features together.

## Step 5 · Train on real recordings, test honestly

> 🟦 **VOCABULARY — Train / test split**
>
> We hide about a third of the recordings, train on the rest, then score the model **only on clips it never heard** — the honest exam. `stratify` keeps healthy and faulty evenly split across both halves.

In [ ]:
X   = feat
idx = np.arange(len(clips))
itr, ite = train_test_split(idx, test_size=0.30, random_state=0, stratify=labels)

model = RandomForestClassifier(n_estimators=300, random_state=0)
model.fit(X.iloc[itr], labels[itr])               # <-- the learning happens here

pred = model.predict(X.iloc[ite])
print('Tested on', len(ite), 'real recordings it never heard.')
print('Accuracy:', round(accuracy_score(labels[ite], pred), 2))

> 🟩 **WHAT JUST HAPPENED**
>
> Around **0.92** on real, unheard recordings — it gets roughly **9 in 10** right. Not perfect, and on *real* data that is the honest truth: factory noise and borderline machines make some calls genuinely hard. A model claiming a flawless 100% on real sound would be the thing to distrust, not to trust.

In [ ]:
print('rows = TRUE, columns = model GUESS   order: [normal, faulty]')
print(confusion_matrix(labels[ite], pred, labels=['normal', 'faulty']))

> 🟩 **WHAT JUST HAPPENED**
>
> The off-diagonal numbers are the misses: a **faulty fan called normal** (one that would slip through) and a **healthy fan called faulty** (a false alarm). On real data both happen — which is why a human stays in the loop on the flagged ones.

## Step 6 · Which part of the sound did it rely on?

A model needn't be a black box. **Feature importance** shows how much it leaned on each number — and we can check that against intuition.

In [ ]:
imp = pd.Series(model.feature_importances_, index=X.columns).sort_values()
plt.figure(figsize=(8, 4.5))
plt.barh(imp.index, imp.values, color='#0b6e7a', edgecolor='white')
plt.xlabel('how much the model relied on each feature')
plt.title('Which part of the sound mattered?'); plt.tight_layout(); plt.show()

> 🟩 **WHAT JUST HAPPENED**
>
> The model leaned hardest on the **higher-frequency bands and brightness** — the whine you heard in the faulty fan. It rediscovered, from the data alone, the very cue an experienced ear uses.

## Step 7 · When you have no fault examples — learn what 'normal' sounds like

> 🟦 **VOCABULARY — Anomaly detection (unsupervised)**
>
> On real equipment, serious faults are **rare** — you may have thousands of hours of *healthy* sound and almost no failures. So flip the question: teach the computer only what **normal** sounds like, then let it flag anything **surprising**. No fault labels needed. It tells you *this is different*, not *what is wrong* — which is exactly when a human should walk over and listen.

In [ ]:
# Train ONLY on healthy recordings -- the detector never sees a single fault.
detector = IsolationForest(n_estimators=300, random_state=0).fit(X[labels == 'normal'])

surprise = -detector.score_samples(X)            # higher = less like a healthy fan
auc = roc_auc_score((labels == 'faulty').astype(int), surprise)
print('Trained on', (labels == 'normal').sum(), 'healthy clips only -- no faults shown.')
print('Healthy-vs-faulty separation (AUC):', round(auc, 2), ' (1.0 = perfect, 0.5 = coin toss)')

> 🟩 **WHAT JUST HAPPENED**
>
> AUC around **0.74** — ranking the odd-sounding fans above the healthy ones works most of the time (a coin toss would be 0.50), but it is clearly **harder** than the labelled version (~0.92): with no examples of what 'faulty' sounds like, the overlap bites. That is the honest reality of condition monitoring on real sound — a useful first filter, not magic. Public MIMII / DCASE fan scores sit in just this range.

In [ ]:
# the alarm line is a DIAL: catch more faults only by tolerating more false alarms
thr = np.percentile(surprise[labels == 'normal'], 80)   # example line (~20% false alarms)
plt.figure(figsize=(9, 5))
plt.hist(surprise[labels == 'normal'], bins=20, alpha=.65, color='#2e7d32', label='normal')
plt.hist(surprise[labels == 'faulty'], bins=20, alpha=.65, color='#b3433a', label='faulty')
plt.axvline(thr, color='#1f2d3d', ls='--', lw=2, label='example alarm line')
plt.xlabel('surprise score  (how UNlike a healthy fan)'); plt.ylabel('number of clips')
plt.title('Learn normal, then flag the surprising'); plt.legend(); plt.grid(alpha=.3)
plt.tight_layout(); plt.show()

print('Where you draw the line is a trade-off:')
for fa_target in [5, 20, 40]:
    t = np.percentile(surprise[labels == 'normal'], 100 - fa_target)
    catch = (surprise[labels == 'faulty'] > t).mean() * 100
    print(f'  tolerate ~{fa_target:>2}% false alarms  ->  catch {catch:>3.0f}% of faulty fans')

> 🟩 **WHAT JUST HAPPENED**
>
> The histograms **overlap** — that is real data — so there is no perfect line. Where you draw it is a **business decision**: tolerate more false alarms to catch more faults, or fewer alarms and miss more. On hard real sound that trade-off is steep, and the printout makes it honest and visible — no free lunch.

## Step 8 · Diagnose a brand-new recording

The payoff: take a real recording the model **never trained on**, play it, and ask for a verdict with a confidence.

In [ ]:
j = next(i for i in ite if labels[i] == 'faulty')   # a held-out faulty fan
print('A new fan is brought in. Listen:')
display(Audio(clips[j], rate=SR))
row   = X.iloc[[j]]
guess = model.predict(row)[0]
conf  = model.predict_proba(row).max()
print(f'The model says:  {guess.upper()}   ({round(conf*100)}% confident)   [true answer: {labels[j].upper()}]')

> 🟩 **WHAT JUST HAPPENED**
>
> The computer heard a real recording it had never encountered and called it — the same thing the seasoned technician does by ear, but it can do it for every machine on the floor, every shift, without tiring.

> 🟧 **YOUR TURN**
>
> Make it your own — change a value and press **Shift + Enter**:
> 1. In the cell above, change `'faulty'` to `'normal'` to diagnose a healthy fan instead — does the model agree?
> 2. In **Step 5**, change `test_size=0.30` to `0.50` (train on fewer clips) and **Runtime → Restart and run all**. Does accuracy drop? Less data usually means a shakier model.
> 3. In **Step 7**, move the alarm line: change `95` to `90` in the `np.percentile(...)` call. You catch more faults — but watch the false-alarm number climb.

> 🟪 **WHAT WE LEARNED**
>
> - A sound is just **numbers** (air-pressure measurements) — so predicting from sound is the same game as predicting from a table.
> - We trained on **real factory recordings** (MIMII / DCASE), not synthetic data — so the score is honest, and there is no hidden formula that trivially separates the classes.
> - A few **features** (energy per frequency band, loudness, brightness) captured what your ear noticed; **feature importance** showed the model leaned on the high-frequency **whine**.
> - **Two ways to learn:** with labelled faults the model *names* the state (~0.92 accuracy); with only healthy sound it *flags the surprising* (anomaly detection, AUC ~0.74) — the realistic case when faults are rare.
> - The familiar rhythm returned: **turn into numbers → split → train → predict → measure**.

> ℹ️ **NOTE**
>
> Where this fits at work: **acoustic condition monitoring** — flagging a failing fan, bearing, gearbox or pump from the noise it makes, before it stops the line. The same recipe extends to vibration sensors and ultrasonic leak detection.

> ℹ️ **NOTE**
>
> **Read frequencies as ORDERS, not raw Hz.** A rotating-machinery engineer will note that a fault's tell-tale pitch moves with shaft speed: imbalance shows at **1×** the rotation rate, misalignment often at **2×**, bearing defects at frequencies set by the bearing geometry. In practice you divide by the running speed so the axis is in *orders* — otherwise simply spinning the machine faster looks like a 'fault'.

> ℹ️ **NOTE**
>
> **The real datasets behind this.** Today's clips come from **MIMII / DCASE** (real factory machine sound for anomaly detection). To go further with real hardware: the **CWRU Bearing** dataset (seeded inner/outer-race and ball faults) and **MAFAULDA** (imbalance, misalignment, bearing — with a microphone channel). The recipe never changes: signal → a few features → classify or flag.

> ℹ️ **NOTE**
>
> **Next — Session 4, the day's finale.** Here the sound *was* the signal and we threw the words away. Next we do the opposite: a machine that turns the **spoken word into text** — and we measure, honestly, how often it mishears the one word that matters.